In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session
!pip install seqeval

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "seqeval", "scikit-learn", "nltk"])

/kaggle/input/datasets/wantodbz/ncbi-disease/validation.csv
/kaggle/input/datasets/wantodbz/ncbi-disease/train.csv
/kaggle/input/datasets/wantodbz/ncbi-disease/test.csv


0

In [ ]:
# === CELL BREAK ===
import pandas as pd
import numpy as np
import ast, re, random, warnings, time, os, json
from collections import Counter, defaultdict
from copy import deepcopy

from seqeval.metrics import (
    f1_score as seqeval_f1,
    precision_score as seqeval_p,
    recall_score as seqeval_r,
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForTokenClassification,
    get_linear_schedule_with_warmup
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('english'))

os.environ["HF_TOKEN"] = ""
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [5]:
# === CELL BREAK ===
# --- KONFIGURASI ---
MODEL_NAME    = 'dmis-lab/biobert-v1.1'
NUM_EPOCHS    = 5
LEARNING_RATE = 5e-5
BATCH_SIZE    = 8
MAX_SEQ_LEN   = 512

TAG2IDX = {'O': 0, 'B-Disease': 1, 'I-Disease': 2}
IDX2TAG = {v: k for k, v in TAG2IDX.items()}
NUM_TAGS = len(TAG2IDX)

GRID_SEED      = 64          # Satu seed untuk grid search
SAMPLING_SEED  = 64

FEW_SHOT_FRACTION = 0.02
FEW_SHOT_LABEL    = '2%'

# --- GRID RS ---
RS_GRID = [5, 10, 15, 20, 25, 30, 35, 40]

OUTPUT_DIR = '/kaggle/working/gridsearch_rs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Scenario      : {FEW_SHOT_LABEL}")
print(f"Grid search   : Rs ∈ {RS_GRID}")
print(f"Seed          : {GRID_SEED}")
print(f"Epochs        : {NUM_EPOCHS}, LR: {LEARNING_RATE}, Batch: {BATCH_SIZE}")

Scenario      : 2%
Grid search   : Rs ∈ [5, 10, 15, 20, 25, 30, 35, 40]
Seed          : 64
Epochs        : 5, LR: 5e-05, Batch: 8


In [6]:
# === CELL BREAK ===
# --- UTILITIES ---
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def parse_sentences(df):
    sentences = []
    for idx, row in df.iterrows():
        tokens_str = row['tokens'].replace("'", '"')
        tokens_str = re.sub(r'\s+', ', ', tokens_str[1:-1])
        tokens_str = '[' + tokens_str + ']'
        try:
            tokens = ast.literal_eval(tokens_str)
        except:
            tokens = re.findall(r"'([^']*)'", row['tokens'])
        tags_str = row['ner_tags'].replace('[', '').replace(']', '')
        tags = [int(t) for t in tags_str.split()]
        iob_tags = []
        for tag in tags:
            if tag == 0:   iob_tags.append('O')
            elif tag == 1: iob_tags.append('B-Disease')
            elif tag == 2: iob_tags.append('I-Disease')
            else:          iob_tags.append('O')
        if len(tokens) == len(iob_tags):
            sentences.append({'id': row['id'], 'tokens': tokens, 'tags': iob_tags})
    return sentences

def sample_few_shot(sentences, fraction, seed=SAMPLING_SEED):
    rng = random.Random(seed)
    n = max(1, int(len(sentences) * fraction))
    indices = list(range(len(sentences)))
    rng.shuffle(indices)
    return [sentences[i] for i in indices[:n]]

# === CELL BREAK ===
# --- iBUS ---
def is_valid_token(token):
    return token.lower() not in STOPWORDS and not (len(token) == 1 and token.isalnum())

def ibus_undersample_sentence(sent, rs):
    tokens, tags = sent['tokens'], sent['tags']
    n_pos = sum(1 for t in tags if t != 'O')
    n_neg = sum(1 for t in tags if t == 'O')
    if n_pos == 0: return None
    if n_neg / n_pos <= rs: return deepcopy(sent)
    maj_req = int(rs * n_pos)
    selected = [t != 'O' for t in tags]
    n_sel = 0
    entities = []
    i = 0
    while i < len(tags):
        if tags[i] != 'O':
            start = i
            while i < len(tags) and tags[i] != 'O': i += 1
            entities.append((start, i))
        else: i += 1

    def select_adj(check_valid=True):
        nonlocal n_sel
        for start, end in entities:
            if n_sel >= maj_req: return
            il = start - 1
            while il >= 0:
                if not selected[il] and tags[il] == 'O':
                    if not check_valid or is_valid_token(tokens[il]):
                        selected[il] = True; n_sel += 1
                    break
                il -= 1
            if n_sel >= maj_req: return
            ir_idx = end
            while ir_idx < len(tokens):
                if not selected[ir_idx] and tags[ir_idx] == 'O':
                    if not check_valid or is_valid_token(tokens[ir_idx]):
                        selected[ir_idx] = True; n_sel += 1
                    break
                ir_idx += 1

    n_inv = sum(1 for i, t in enumerate(tags) if t == 'O' and not is_valid_token(tokens[i]))
    if maj_req <= (n_neg - n_inv):
        while n_sel < maj_req:
            prev = n_sel; select_adj(True)
            if n_sel == prev: break
        for i in range(len(tokens)):
            if n_sel >= maj_req: break
            if not selected[i] and tags[i] == 'O' and is_valid_token(tokens[i]):
                selected[i] = True; n_sel += 1
    else:
        while n_sel < min(maj_req, n_neg - n_inv):
            prev = n_sel; select_adj(True)
            if n_sel == prev: break
        for i in range(len(tokens)):
            if n_sel >= maj_req: break
            if not selected[i] and tags[i] == 'O' and is_valid_token(tokens[i]):
                selected[i] = True; n_sel += 1
        while n_sel < maj_req:
            prev = n_sel; select_adj(False)
            if n_sel == prev: break
        for i in range(len(tokens)):
            if n_sel >= maj_req: break
            if not selected[i] and tags[i] == 'O':
                selected[i] = True; n_sel += 1

    return {
        'id': sent['id'],
        'tokens': [t for t, s in zip(tokens, selected) if s],
        'tags':   [t for t, s in zip(tags, selected) if s]
    }

def ibus_undersample_dataset(sentences, rs):
    result = []
    for sent in sentences:
        us = ibus_undersample_sentence(sent, rs)
        result.append(us if us is not None else deepcopy(sent))
    return result

def compute_dataset_ir(sentences):
    """Hitung rata-rata IR dataset setelah undersampling."""
    irs = []
    for sent in sentences:
        n_pos = sum(1 for t in sent['tags'] if t != 'O')
        n_neg = sum(1 for t in sent['tags'] if t == 'O')
        if n_pos > 0:
            irs.append(n_neg / n_pos)
    return float(np.mean(irs)) if irs else 0.0

# === CELL BREAK ===
# --- MODEL & TRAINING ---
class NERDatasetBERT(Dataset):
    def __init__(self, sentences, tokenizer, tag2idx, max_len=MAX_SEQ_LEN):
        self.sentences = sentences
        self.tokenizer = tokenizer
        self.tag2idx = tag2idx
        self.max_len = max_len

    def __len__(self): return len(self.sentences)

    def __getitem__(self, idx):
        sent = self.sentences[idx]
        tokens, tags = sent['tokens'], sent['tags']
        encoding = self.tokenizer(
            tokens, is_split_into_words=True,
            max_length=self.max_len, padding='max_length',
            truncation=True, return_tensors='pt'
        )
        word_ids = encoding.word_ids()
        aligned = []; prev = None
        for wid in word_ids:
            if wid is None: aligned.append(-100)
            elif wid != prev:
                aligned.append(self.tag2idx[tags[wid]] if wid < len(tags) else -100)
            else:
                if wid < len(tags):
                    tag = tags[wid]
                    if tag.startswith('B-'): aligned.append(self.tag2idx['I-' + tag[2:]])
                    else: aligned.append(self.tag2idx[tag])
                else: aligned.append(-100)
            prev = wid
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(aligned, dtype=torch.long)
        }

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def evaluate_model(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch in loader:
            ids   = batch['input_ids'].to(device)
            mask  = batch['attention_mask'].to(device)
            labels = batch['labels']
            preds  = model(input_ids=ids, attention_mask=mask).logits.argmax(dim=-1).cpu().tolist()
            for ps, ls in zip(preds, labels.tolist()):
                pt, tt = [], []
                for p, l in zip(ps, ls):
                    if l != -100:
                        pt.append(IDX2TAG.get(p, 'O'))
                        tt.append(IDX2TAG.get(l, 'O'))
                if tt: y_true.append(tt); y_pred.append(pt)
    return seqeval_f1(y_true, y_pred), seqeval_p(y_true, y_pred), seqeval_r(y_true, y_pred)

def train_single(train_sents, val_sents, test_sents, seed, label=""):
    """
    Train + evaluasi satu run.
    Return: dict berisi val_f1, test_f1, test_p, test_r, best_epoch, time
    """
    set_seed(seed)
    train_ds    = NERDatasetBERT(train_sents, tokenizer, TAG2IDX)
    val_ds      = NERDatasetBERT(val_sents,   tokenizer, TAG2IDX)
    test_ds     = NERDatasetBERT(test_sents,  tokenizer, TAG2IDX)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*4, shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE*4, shuffle=False)

    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_TAGS
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, 0, len(train_loader) * NUM_EPOCHS
    )

    best_val_f1, best_state, best_epoch = 0.0, None, 0
    t0 = time.time()

    for epoch in range(NUM_EPOCHS):
        model.train(); total_loss = 0
        for batch in train_loader:
            ids    = batch['input_ids'].to(device)
            mask   = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            loss   = model(input_ids=ids, attention_mask=mask, labels=labels).loss
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        vf1, vp, vr = evaluate_model(model, val_loader)
        print(f"    [{label}] Epoch {epoch+1}/{NUM_EPOCHS}: loss={avg_loss:.4f}, val_F1={vf1:.4f}")

        if vf1 > best_val_f1:
            best_val_f1 = vf1
            best_state  = deepcopy(model.state_dict())
            best_epoch  = epoch + 1

    # Evaluasi test menggunakan checkpoint val terbaik
    if best_state:
        model.load_state_dict(best_state)
    tf1, tp, tr = evaluate_model(model, test_loader)
    elapsed = time.time() - t0

    print(f"    [{label}] >> Test: F1={tf1:.4f}, P={tp:.4f}, R={tr:.4f} | Best val_F1={best_val_f1:.4f} @ epoch {best_epoch} | Time={elapsed:.1f}s")
    del model; torch.cuda.empty_cache()

    return {
        'val_f1': best_val_f1,
        'test_f1': tf1, 'test_p': tp, 'test_r': tr,
        'best_epoch': best_epoch,
        'time': elapsed
    }

config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [7]:
# === CELL BREAK ===
# --- LOAD DATA ---
print("="*65)
print(f"LOADING DATA — {FEW_SHOT_LABEL}")
print("="*65)

train_df = pd.read_csv('/kaggle/input/datasets/wantodbz/ncbi-disease/train.csv')
val_df   = pd.read_csv('/kaggle/input/datasets/wantodbz/ncbi-disease/validation.csv')
test_df  = pd.read_csv('/kaggle/input/datasets/wantodbz/ncbi-disease/test.csv')

train_sentences = parse_sentences(train_df)
val_sentences   = parse_sentences(val_df)
test_sentences  = parse_sentences(test_df)

fs_train = sample_few_shot(train_sentences, FEW_SHOT_FRACTION)
n_ent    = sum(1 for s in fs_train for t in s['tags'] if t.startswith('B-'))
print(f"Few-shot {FEW_SHOT_LABEL}: {len(fs_train)} sentences, {n_ent} entities")
print(f"Val: {len(val_sentences)} sentences | Test: {len(test_sentences)} sentences")

LOADING DATA — 2%
Few-shot 2%: 107 sentences, 89 entities
Val: 919 sentences | Test: 938 sentences


In [8]:
 
# --- GRID SEARCH LOOP ---
print("\n" + "="*65)
print(f"GRID SEARCH iBUS Rs — seed={GRID_SEED}")
print("="*65)

gs_records = []

for rs in RS_GRID:
    print(f"\n{'─'*55}")
    print(f"Rs = {rs}")
    print(f"{'─'*55}")

    # Terapkan iBUS
    ibus_data = ibus_undersample_dataset(fs_train, rs)
    mean_ir   = compute_dataset_ir(ibus_data)
    n_tokens  = sum(len(s['tokens']) for s in ibus_data)
    n_pos_tok = sum(1 for s in ibus_data for t in s['tags'] if t != 'O')
    print(f"  Sentences : {len(ibus_data)}")
    print(f"  Tokens    : {n_tokens} (positif: {n_pos_tok})")
    print(f"  Mean IR   : {mean_ir:.2f}")

    label = f"iBUS_Rs{rs}"
    res   = train_single(fs_train_ibus := ibus_data,
                         val_sentences, test_sentences,
                         seed=GRID_SEED, label=label)

    gs_records.append({
        'Rs'         : rs,
        'n_sentences': len(ibus_data),
        'n_tokens'   : n_tokens,
        'n_pos_tokens': n_pos_tok,
        'mean_ir'    : round(mean_ir, 2),
        'val_f1'     : round(res['val_f1'], 4),
        'test_f1'    : round(res['test_f1'], 4),
        'test_p'     : round(res['test_p'], 4),
        'test_r'     : round(res['test_r'], 4),
        'best_epoch' : res['best_epoch'],
        'time_sec'   : round(res['time'], 1),
    })
    print(f"  val_F1={res['val_f1']:.4f} | test_F1={res['test_f1']:.4f}")

# === CELL BREAK ===
# --- SUMMARY & PILIH RS TERBAIK ---
print("\n" + "="*65)
print("GRID SEARCH SUMMARY")
print("="*65)

df_gs = pd.DataFrame(gs_records).sort_values('val_f1', ascending=False)
print(df_gs[['Rs', 'n_sentences', 'mean_ir',
             'val_f1', 'test_f1', 'test_p', 'test_r',
             'best_epoch', 'time_sec']].to_string(index=False))

best_row = df_gs.iloc[0]
best_rs  = int(best_row['Rs'])
print(f"\n★ Rs terbaik (berdasar val_F1): Rs={best_rs}")
print(f"  val_F1={best_row['val_f1']:.4f} | test_F1={best_row['test_f1']:.4f}")

# Simpan hasil
df_gs.to_csv(os.path.join(OUTPUT_DIR, 'gridsearch_rs_results.csv'), index=False)
print(f"Hasil disimpan → {OUTPUT_DIR}/gridsearch_rs_results.csv")

# === CELL BREAK ===
# --- VISUALISASI ---
print("\n" + "="*65)
print("GENERATING PLOTS")
print("="*65)

df_plot = pd.DataFrame(gs_records).sort_values('Rs')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Grid Search iBUS Rs — seed={GRID_SEED}, {FEW_SHOT_LABEL}',
             fontsize=14, fontweight='bold')

# Plot 1: Val F1 vs Rs
ax = axes[0]
ax.plot(df_plot['Rs'], df_plot['val_f1'], 'o-', color='#3498db',
        linewidth=2, markersize=8, label='Val F1')
ax.plot(df_plot['Rs'], df_plot['test_f1'], 's--', color='#e74c3c',
        linewidth=2, markersize=8, label='Test F1')
ax.axvline(best_rs, color='#27ae60', linestyle=':', linewidth=1.5,
           label=f'Best Rs={best_rs}')
ax.set_xlabel('Rs'); ax.set_ylabel('F1 Score')
ax.set_title('Val & Test F1 vs Rs')
ax.legend(); ax.grid(alpha=0.3)
for _, row in df_plot.iterrows():
    ax.annotate(f"{row['val_f1']:.3f}",
                (row['Rs'], row['val_f1']),
                textcoords="offset points", xytext=(0, 8),
                ha='center', fontsize=8)

# Plot 2: Mean IR setelah undersampling
ax = axes[1]
ax.bar(df_plot['Rs'].astype(str), df_plot['mean_ir'],
       color='#9b59b6', edgecolor='black', linewidth=0.5, alpha=0.8)
ax.set_xlabel('Rs'); ax.set_ylabel('Mean IR')
ax.set_title('Mean IR Dataset setelah iBUS')
ax.grid(axis='y', alpha=0.3)
for i, (_, row) in enumerate(df_plot.iterrows()):
    ax.text(i, row['mean_ir'] + 0.2, f"{row['mean_ir']:.1f}",
            ha='center', fontsize=9)

# Plot 3: P / R / F1 per Rs (heatmap style)
ax = axes[2]
prf_data = df_plot[['Rs', 'test_p', 'test_r', 'test_f1']].set_index('Rs')
im = ax.imshow(prf_data.T.values, aspect='auto', cmap='YlGn',
               vmin=0, vmax=1)
ax.set_xticks(range(len(prf_data)))
ax.set_xticklabels(prf_data.index.tolist())
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(['Precision', 'Recall', 'F1'])
ax.set_title('Test P/R/F1 Heatmap per Rs')
ax.set_xlabel('Rs')
for i in range(3):
    for j in range(len(prf_data)):
        ax.text(j, i, f"{prf_data.values[j, i]:.3f}",
                ha='center', va='center', fontsize=8, color='black')
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'gridsearch_rs_plot.png'),
            dpi=300, bbox_inches='tight')
plt.show(); plt.close()
print("Plot disimpan.")

# === CELL BREAK ===
print("\n" + "="*65)
print("GRID SEARCH SELESAI")
print("="*65)
print(f"\nRs terbaik : {best_rs}")
print(f"Val F1     : {best_row['val_f1']:.4f}")
print(f"Test F1    : {best_row['test_f1']:.4f}")
print(f"Test P     : {best_row['test_p']:.4f}")
print(f"Test R     : {best_row['test_r']:.4f}")
print(f"\nOutput dir : {OUTPUT_DIR}")


GRID SEARCH iBUS Rs — seed=64

───────────────────────────────────────────────────────
Rs = 5
───────────────────────────────────────────────────────
  Sentences : 107
  Tokens    : 2110 (positif: 186)
  Mean IR   : 4.18


pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-v1.1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

...

BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-v1.1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    [iBUS_Rs35] Epoch 1/5: loss=0.4220, val_F1=0.0000
    [iBUS_Rs35] Epoch 2/5: loss=0.1809, val_F1=0.3738
    [iBUS_Rs35] Epoch 3/5: loss=0.0818, val_F1=0.6102
    [iBUS_Rs35] Epoch 4/5: loss=0.0378, val_F1=0.5728
    [iBUS_Rs35] Epoch 5/5: loss=0.0185, val_F1=0.6139
    [iBUS_Rs35] >> Test: F1=0.6328, P=0.6456, R=0.6205 | Best val_F1=0.6139 @ epoch 5 | Time=227.8s
  val_F1=0.6139 | test_F1=0.6328

───────────────────────────────────────────────────────
Rs = 40
───────────────────────────────────────────────────────
  Sentences : 107
  Tokens    : 2619 (positif: 186)
  Mean IR   : 12.02


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-v1.1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    [iBUS_Rs40] Epoch 1/5: loss=0.4169, val_F1=0.0125
    [iBUS_Rs40] Epoch 2/5: loss=0.1686, val_F1=0.3721
    [iBUS_Rs40] Epoch 3/5: loss=0.0758, val_F1=0.6389
    [iBUS_Rs40] Epoch 4/5: loss=0.0383, val_F1=0.5686
    [iBUS_Rs40] Epoch 5/5: loss=0.0200, val_F1=0.6008
    [iBUS_Rs40] >> Test: F1=0.6587, P=0.6294, R=0.6908 | Best val_F1=0.6389 @ epoch 3 | Time=227.4s
  val_F1=0.6389 | test_F1=0.6587

GRID SEARCH SUMMARY
 Rs  n_sentences  mean_ir  val_f1  test_f1  test_p  test_r  best_epoch  time_sec
 15          107     8.60  0.6499   0.6537  0.6222  0.6887           3     227.9
 25          107    10.46  0.6411   0.6680  0.6422  0.6960           3     228.0
 40          107    12.02  0.6389   0.6587  0.6294  0.6908           3     227.4
 30          107    11.06  0.6258   0.6277  0.6429  0.6132           5     227.7
 10          107     6.73  0.6222   0.6381  0.6219  0.6551           3     228.0
 20          107     9.69  0.6140   0.6288  0.6429  0.6153           5     228.3
 35      